In [1]:
!pip install xgboost lightgbm scikit-learn


   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   - -------------------------------------- 0.0/1.5 MB 653.6 kB/s eta 0:00:03
   --- ------------------------------------ 0.1/1.5 MB 1.0 MB/s eta 0:00:02
   ------- -------------------------------- 0.3/1.5 MB 1.6 MB/s eta 0:00:01
   ----------- ---------------------------- 0.4/1.5 MB 2.0 MB/s eta 0:00:01
   --------------- ------------------------ 0.6/1.5 MB 2.5 MB/s eta 0:00:01
   ---------------- ----------------------- 0.6/1.5 MB 2.3 MB/s eta 0:00:01
   ---------------- ----------------------- 0.6/1.5 MB 2.3 MB/s eta 0:00:01
   ---------------- ----------------------- 0.6/1.5 MB 2.3 MB/s eta 0:00:01
   ---------------- ----------------------- 0.6/1.5 MB 2.3 MB/s eta 0:00:01
   -------------------- ------------------- 0.7/1.5 MB 1.6 MB/s eta 0:00:01
   ---------------------- ----------------- 0.8/1.5 MB 1.6 MB/s eta 0:00:01
   ---------------------


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import torch
import numpy as np
import time

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier


In [3]:
data = torch.load("embeddings.pth")

X_train = data["train"]["embeddings"].cpu().numpy()
y_train = data["train"]["labels"].cpu().numpy()

X_val = data["val"]["embeddings"].cpu().numpy()
y_val = data["val"]["labels"].cpu().numpy()

X_test = data["test"]["embeddings"].cpu().numpy()
y_test = data["test"]["labels"].cpu().numpy()

print("Train:", X_train.shape)
print("Val:", X_val.shape)
print("Test:", X_test.shape)


Train: (89996, 512)
Val: (10004, 512)
Test: (7180, 512)


In [4]:
y_train = y_train.reshape(-1)
y_val = y_val.reshape(-1)
y_test = y_test.reshape(-1)


In [5]:
start = time.time()

xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='multi:softmax',
    eval_metric='mlogloss',
    num_class=len(np.unique(y_train)),
    random_state=42
)

xgb_model.fit(X_train, y_train)

train_time = time.time() - start
print("Training time (XGBoost):", round(train_time,2), "seconds")


Training time (XGBoost): 1009.54 seconds


In [6]:
y_pred_xgb = xgb_model.predict(X_test)

acc_xgb = accuracy_score(y_test, y_pred_xgb)
print("XGBoost Test Accuracy:", round(acc_xgb*100,2), "%")

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_xgb))


XGBoost Test Accuracy: 88.51 %

Classification Report:

              precision    recall  f1-score   support

           0       0.99      0.99      0.99      1338
           1       0.97      1.00      0.99       847
           2       0.68      0.71      0.69       339
           3       0.95      0.93      0.94       634
           4       0.96      0.93      0.94      1035
           5       0.71      0.74      0.72       592
           6       0.87      0.86      0.86       741
           7       0.59      0.52      0.55       421
           8       0.86      0.90      0.88      1233

    accuracy                           0.89      7180
   macro avg       0.84      0.84      0.84      7180
weighted avg       0.88      0.89      0.88      7180



In [7]:
cm_xgb = confusion_matrix(y_test, y_pred_xgb)
print("Confusion Matrix:\n", cm_xgb)


Confusion Matrix:
 [[1319    0    1    0    6    8    4    0    0]
 [   0  847    0    0    0    0    0    0    0]
 [   0    0  240    0    0   72    0   23    4]
 [   0    0    7  587    0    0   18    2   20]
 [  11   24    7    1  960    2   12   11    7]
 [   0    0   23    0    3  436    0  115   15]
 [   1    0    1    9   11    3  636    0   80]
 [   0    0   44    0    8   88    7  221   53]
 [   0    0   32   18    9    7   54    4 1109]]


In [8]:
start = time.time()

lgb_model = LGBMClassifier(
    n_estimators=400,
    learning_rate=0.05,
    max_depth=-1,
    random_state=42
)

lgb_model.fit(X_train, y_train)

train_time = time.time() - start
print("Training time (LightGBM):", round(train_time,2), "seconds")


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.517365 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 130560
[LightGBM] [Info] Number of data points in the train set: 89996, number of used features: 512
[LightGBM] [Info] Start training from score -2.262679
[LightGBM] [Info] Start training from score -2.247527
[LightGBM] [Info] Start training from score -2.161813
[LightGBM] [Info] Start training from score -2.157863
[LightGBM] [Info] Start training from score -2.419574
[LightGBM] [Info] Start training from score -1.999806
[LightGBM] [Info] Start training from score -2.434676
[LightGBM] [Info] Start training from score -2.258949
[LightGBM] [Info] Start training from score -1.943701
Training time (LightGBM): 440.26 seconds


In [9]:
y_pred_lgb = lgb_model.predict(X_test)

acc_lgb = accuracy_score(y_test, y_pred_lgb)
print("LightGBM Test Accuracy:", round(acc_lgb*100,2), "%")

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_lgb))

cm_lgb = confusion_matrix(y_test, y_pred_lgb)
print("Confusion Matrix:\n", cm_lgb)


D:\python\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


LightGBM Test Accuracy: 89.58 %

Classification Report:

              precision    recall  f1-score   support

           0       0.99      0.98      0.99      1338
           1       0.98      1.00      0.99       847
           2       0.72      0.78      0.75       339
           3       0.95      0.94      0.94       634
           4       0.97      0.94      0.95      1035
           5       0.74      0.76      0.75       592
           6       0.89      0.87      0.88       741
           7       0.61      0.53      0.57       421
           8       0.87      0.91      0.89      1233

    accuracy                           0.90      7180
   macro avg       0.86      0.86      0.86      7180
weighted avg       0.90      0.90      0.90      7180

Confusion Matrix:
 [[1313    0    1    0    7   12    4    0    1]
 [   1  846    0    0    0    0    0    0    0]
 [   0    0  265    0    0   49    0   20    5]
 [   0    0    5  594    0    0   13    2   20]
 [  10   20    5    0  971 